In [ ]:
import logging

logging.basicConfig(level="DEBUG")
logging.getLogger("fsspec").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [ ]:
import os
import torch

from darts.utils.likelihood_models import QuantileRegression

# must use pytorch_lightning style because darts is using that internally
from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning import seed_everything
from pytorch_lightning.callbacks import EarlyStopping
from darts.models import RNNModel
import torchmetrics
from torchmetrics import MetricCollection

from darts.dataprocessing.transformers import Scaler
from sklearn.preprocessing import StandardScaler

from aare.feature_set import FeatureSet
from aare.features.registry import FEATURES

from aare.evaluation.evaluation import evaluate_model
from aare.params import read_params

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()

In [ ]:
import mlflow

mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

In [ ]:
RANDOM_SEED = 42
seed_everything(RANDOM_SEED)

# RNN Models (GRU)

This can finally model the non-linear relationships. Although my research suggests that there are only few long-term factors, it might still be a good model, let's just try.


In [ ]:
model_name = "GRU"

In [ ]:
validation_params = params["validation"]
stride = validation_params["stride"]
min_lookback_hours = validation_params["min_lookback_hours"]
forecast_horizon = params["general"]["forecast_horizon"]

In [ ]:
features = {
    "targets": ["temp_bern"],
    "future": [
        "tt_bern",
        "flow_bern",
        "ss_bern",
    ],
}

In [ ]:
ds = FeatureSet(
    targets=[FEATURES[f] for f in features["targets"]],
    future=[FEATURES[f] for f in features["future"]],
    split_params=params["split"],
)

In [ ]:
train = ds.get_train()
train

In [ ]:
val = ds.get_val()
val

In [ ]:
train_target_subs = train[0]
train_fc_subs = train[2]
val_target_subs = val[0]
val_fc_subs = val[2]

In [ ]:
[len(x) for x in train_target_subs]

In [ ]:
train_lens = [len(x) for x in train_target_subs]
val_lens = [len(x) for x in val_target_subs]
data_stats = {
    "train_lens": train_lens,
    "train_len_total": sum(train_lens),
    "train_n_subs": len(train_lens),
    "val_lens": val_lens,
    "val_len_total": sum(val_lens),
    "val_n_subs": len(val_lens),
    "val_split": sum(val_lens) / (sum(val_lens) + sum(train_lens)),
}

In [ ]:
scaler_target = Scaler(StandardScaler(), global_fit=True)
scaler_fc = Scaler(StandardScaler(), global_fit=True)

In [ ]:
scaler_target.fit(train_target_subs)
scaler_fc.fit(train_fc_subs)

In [ ]:
from aare.evaluation.evaluation import DataTransformers

data_transformers: DataTransformers = {
    "series": scaler_target,
    "future_covariates": scaler_fc,
}

In [ ]:
mlflow.end_run()
mlflow.set_experiment(model_name)
run = mlflow.start_run(log_system_metrics=True)

In [ ]:
# seems like the default doesn't work?!? it should take that without setting it explicitly..
# https://github.com/Lightning-AI/pytorch-lightning/discussions/11197#discussioncomment-9164713
mlflow_logger = MLFlowLogger(model_name, tracking_uri=os.getenv("MLFLOW_TRACKING_URI"), run_id=run.info.run_id)

In [ ]:
early_stopping = EarlyStopping("val_loss", patience=5)

In [ ]:
# https://unit8co.github.io/darts/examples/08-DeepAR-examples.html
input_chunk_length = 1  # how many hours lookback when predicting
hparams_model = dict(
    model=model_name,
    input_chunk_length=input_chunk_length,
    hidden_dim=128,
    dropout=0.1,
    n_rnn_layers=3,
    # output_chunk_length is always 1 for RNNs, but at inference time,
    # we want to predict more than 1 data point at once. To better model
    # that behaviour, it takes a training_length and will do
    # training_length - input_chunk_length (= forecast_horizon) steps
    # and combine the loss of all of them before optimizing.
    # https://github.com/unit8co/darts/issues/1397#issuecomment-1331936411
    training_length=forecast_horizon + input_chunk_length,
    # training kwargs
    model_name=model_name,
    random_state=RANDOM_SEED,
    batch_size=1024,
    n_epochs=100,
    optimizer_cls=torch.optim.Adam,
    optimizer_kwargs=dict(
        lr=1e-5,
    ),
    # lr_scheduler_cls=
    # lr_scheduler_kwargs=
    # either loss or likelihood
    # loss_fn=nn.MSELoss(),
    likelihood=QuantileRegression([0.25, 0.5, 0.75]),
    torch_metrics=MetricCollection(
        {
            "mae": torchmetrics.MeanAbsoluteError(),
            "rmse": torchmetrics.MeanSquaredError(squared=False),
        }
    ),
    save_checkpoints=False,
    pl_trainer_kwargs={"logger": mlflow_logger, "callbacks": [early_stopping], "log_every_n_steps": 50},
    add_encoders={"cyclic": {"future": ["hour", "day_of_year"]}},
)

In [ ]:
model = RNNModel(**hparams_model)

In [ ]:
model.fit(
    series=scaler_target.transform(train_target_subs),
    future_covariates=scaler_fc.transform(train_fc_subs),
    val_series=scaler_target.transform(val_target_subs),
    val_future_covariates=scaler_fc.transform(val_fc_subs),
)

In [ ]:
metrics, samples = evaluate_model(
    model,
    val_target_subs,
    forecast_horizon,
    stride,
    min_lookback_hours,
    future_cov=val_fc_subs,
    data_transformers=data_transformers,
)

In [ ]:
metrics

In [ ]:
_ = samples.plot(model_name, with_covariates=False)

In [ ]:
hparams = (
    {
        "horizon": forecast_horizon,
        "val_stride": stride,
        "split_train": params["split"]["train_split"],
        "split_val": params["split"]["val_split"],
        "split_test": params["split"]["test_split"],
        "features_targets": features["targets"],
        "features_future": features["future"],
    }
    | {"model_" + key: value for key, value in hparams_model.items()}
    | {"data_" + key: value for key, value in data_stats.items()}
)

In [ ]:
mlflow.log_params(hparams)
mlflow.log_metrics({f"eval_{k}": v for k, v in metrics.to_dict().items()})

In [ ]:
mlflow.end_run()